# Keyword study 3: results

Run `python scripts/kw_analyze.py --config configs/kw_week1.toml` first; this notebook reads its tables and adds examples.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display
from fairsum.kwsum.collections import read_collections
from fairsum.kwsum.config import COLLECTIONS_PATH, KEYWORDS_PATH, MANIFEST_PATH, POOLS_DIR, load_config
from fairsum.kwsum.store import read_jsonl

cfg = load_config("../configs/kw_week1.toml")
pd.set_option("display.max_colwidth", 200)
from fairsum.kwsum.stats import compliance_table, condition_means, run_tests
scores = pd.DataFrame(read_jsonl(cfg.scores_path))
tables = cfg.results_dir / 'tables'
figures = cfg.results_dir / 'figures'

## Compliance: report beside every result

In [ ]:
pd.read_csv(tables / 'compliance.csv')

## Primary metrics by condition

In [ ]:
means = pd.read_csv(tables / 'means.csv')
primary = ['unsupported_rate', 'contradiction_rate', 'count_abs_error', 'delta_p', 'majority_amplification', 'coverage_gap']
means.set_index(['model', 'dataset', 'condition'])[primary + ['n_scored']].round(3)

## Registered hypotheses (Holm-corrected within model x dataset)

In [ ]:
pd.read_csv(tables / 'hypotheses.csv')[['model', 'dataset', 'hypothesis', 'comparison', 'metric', 'n', 'mean_diff', 'ci_lo', 'ci_hi', 'p_holm', 'supported']].round(4)

## Figures

In [ ]:
display(Image(filename=str(figures / 'majority_amplification.png')))
display(Image(filename=str(figures / 'faithfulness_fairness_tradeoff.png')))

## Examples: the largest majority amplification from global keywords at 7:1
For the paper's qualitative figure. Read these before believing any mean.

In [ ]:
from fairsum.kwsum.stats import aligned_pairs
gens = pd.DataFrame(read_jsonl(cfg.generations_path)).set_index(['collection_id', 'condition', 'model'])
pairs = aligned_pairs(scores, 'majority_amplification', 'kw_global', 'direct', levels=['pos88', 'neg88']).sort_values('diff', ascending=False)
for _, p in pairs.head(3).iterrows():
    print(f"=== {p['collection_id']} | {p['model']} | MA diff {p['diff']:.2f}")
    for cond in ('direct', 'kw_global', 'kw_strat_neg'):
        key = (p['collection_id'], cond, p['model'])
        if key in gens.index:
            print(f'[{cond}]', gens.loc[key, 'raw'][:600], '\n')